# DE Gene Overlap & Enrichment Analysis

This notebook picks up where the **atlas DE analysis** notebook left off.

**Workflow:**
1. Load per-comparison DE result tables (CSV files produced by `atlas_de_analysis.ipynb`)
2. Filter to significantly **upregulated** genes per comparison
3. Visualise overlaps with an **UpSet plot** (`scutils.pl.upset_plot`)
4. Build a **consensus gene set** (genes present in ≥ a configurable fraction of comparisons)
5. Run **over-representation analysis** (ORA) against multiple gene-set resources using `decoupler`
6. Export the consensus gene list and enrichment tables

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import decoupler as dc
import matplotlib.pyplot as plt
import pandas as pd

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `RESULTS_DIR` — path to the folder containing per-comparison CSV files
  produced by `atlas_de_analysis.ipynb` (e.g. `de_Keratinocyte_Tumor_vs_Normal.csv`)
- `LFC_CUTOFF` / `PVAL_CUTOFF` — thresholds for selecting upregulated genes
- `MIN_OVERLAP_FRACTION` — fraction of comparisons a gene must be
  upregulated in to enter the consensus set.  `0.0` → union of all genes;
  `1.0` → strict intersection; intermediate values (e.g. `0.75`) give a
  robust consensus allowing a gene to be missing from a few comparisons
- `GENE_SET_RESOURCES` — gene-set databases to query for ORA.  Built-in
  decoupler resources (Hallmark, PROGENy, CollecTRI, DoRothEA) and MSigDB
  collections from OmniPath (GO, KEGG, Reactome, WikiPathways) are
  supported out of the box.  Remove entries you don't need

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
# Directory containing per-comparison CSV files from atlas_de_analysis.ipynb
# Expected file pattern: de_<label>.csv  (one per comparison)
RESULTS_DIR = Path("results/de_atlas")

# ── Significance thresholds for selecting upregulated genes ───────────────────
PVAL_CUTOFF = 0.05    # adjusted p-value threshold
LFC_CUTOFF  = 0.5     # minimum log2FoldChange (> 0 = upregulated)

# ── Minimum DE genes per comparison ──────────────────────────────────────────
# Comparisons with fewer upregulated genes than this threshold are skipped.
# Set to 0 to keep all comparisons regardless of how many DE genes they yield.
MIN_DE_GENES = 0

# ── Consensus overlap sensitivity ─────────────────────────────────────────────
# Fraction of comparisons a gene must be upregulated in to be included in
# the consensus gene set.
#   0.0  → union (gene appears in ≥ 1 comparison)
#   0.5  → gene must be significant in ≥ 50 % of comparisons
#   1.0  → strict intersection (gene must be in ALL comparisons)
MIN_OVERLAP_FRACTION = 0.75

# ── Gene-set resources for over-representation analysis ───────────────────────
# Each entry maps a display name to a resource key.
#
# • Keys recognised by the built-in decoupler getters:
#     hallmark, progeny, collectri, dorothea
#
# • Keys prefixed with a MSigDB collection name (fetched via OmniPath):
#     go_biological_process, go_cellular_component, go_molecular_function,
#     kegg_pathways, reactome_pathways, wikipathways
#
# Comment out or remove entries you do not need.
GENE_SET_RESOURCES: dict[str, str] = {
    # ── Regulatory / signalling resources (decoupler built-ins) ───────────
    "MSigDB Hallmark": "hallmark",
    "PROGENy":         "progeny",
    "CollecTRI":       "collectri",
    "DoRothEA":        "dorothea",
    # ── Pathway / ontology databases (MSigDB via OmniPath) ────────────────
    "GO Biological Process": "go_biological_process",
    "GO Cellular Component": "go_cellular_component",
    "GO Molecular Function": "go_molecular_function",
    "KEGG Pathways":         "kegg_pathways",
    "Reactome":              "reactome_pathways",
    "WikiPathways":          "wikipathways",
}

# Organism for gene-set retrieval
ORGANISM = "human"

# Adjusted p-value threshold for enrichment results
ENRICH_PVAL_CUTOFF = 0.05

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("results/de_overlap")

print("Configuration loaded.")
print(f"  Results dir          : {RESULTS_DIR}")
print(f"  LFC cutoff           : {LFC_CUTOFF}")
print(f"  p-value cutoff       : {PVAL_CUTOFF}")
print(f"  Min DE genes         : {MIN_DE_GENES}")
print(f"  Overlap fraction     : {MIN_OVERLAP_FRACTION}")
print(f"  Gene-set resources   : {list(GENE_SET_RESOURCES.keys())}")
print(f"  Enrichment p cutoff  : {ENRICH_PVAL_CUTOFF}")
print(f"  Output dir           : {OUTPUT_DIR}")

## 3. Load DE Results & Filter Upregulated Genes

Read each per-comparison CSV produced by the atlas DE notebook, filter to
genes that are both significantly differentially expressed (`padj < PVAL_CUTOFF`)
and upregulated (`log2FoldChange > LFC_CUTOFF`), and store one set of gene
names per comparison.

In [ ]:
# ── Discover per-comparison files ─────────────────────────────────────────────
csv_files = sorted(RESULTS_DIR.glob("de_*.csv"))
# Exclude the combined file if present
csv_files = [f for f in csv_files if "all_comparisons" not in f.name]

if not csv_files:
    raise FileNotFoundError(
        f"No per-comparison CSV files found in {RESULTS_DIR}. "
        f"Run the atlas DE notebook first."
    )

print(f"Found {len(csv_files)} comparison file(s):")
for f in csv_files:
    print(f"  • {f.name}")

# ── Load and filter ──────────────────────────────────────────────────────────
upreg_sets: dict[str, set[str]] = {}
skipped: list[str] = []

for csv_path in csv_files:
    # Derive a short label from filename: de_<label>.csv → <label>
    label = csv_path.stem.removeprefix("de_")

    df = pd.read_csv(csv_path)

    # Filter: significantly upregulated
    mask = (df["padj"] < PVAL_CUTOFF) & (df["log2FoldChange"] > LFC_CUTOFF)
    genes = set(df.loc[mask, "gene"].dropna().unique())

    if len(genes) <= MIN_DE_GENES:
        skipped.append(label)
        print(f"  {label}: {len(genes)} upregulated genes  ⚠ SKIPPED (≤ {MIN_DE_GENES})")
        continue

    upreg_sets[label] = genes
    print(f"  {label}: {len(genes)} upregulated genes")

if skipped:
    print(f"\n⚠  Skipped {len(skipped)} comparison(s) with ≤ {MIN_DE_GENES} DE genes: {skipped}")

n_comparisons = len(upreg_sets)
if n_comparisons == 0:
    raise RuntimeError(
        "No comparisons passed the MIN_DE_GENES filter. "
        "Consider lowering MIN_DE_GENES or adjusting PVAL_CUTOFF / LFC_CUTOFF."
    )

all_genes = set().union(*upreg_sets.values())
print(f"\n{n_comparisons} comparison(s) retained with {len(all_genes)} unique upregulated genes total.")

## 4. UpSet Plot — Visualise Overlaps

The UpSet plot shows the size of every pairwise and higher-order
intersection across comparisons.  Each bar represents a group of genes
shared by a specific combination of comparisons.

In [ ]:
fig = scutils.pl.upset_plot(
    upreg_sets,
    figsize=(12, 6),
    sort_by="cardinality",
    show_counts=True,
    min_subset_size=1,
)
fig.suptitle(
    f"Overlap of upregulated genes  "
    f"(padj < {PVAL_CUTOFF}, log2FC > {LFC_CUTOFF})",
    fontsize=12,
    y=1.02,
)

if OUTPUT_DIR is not None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / "upset_upregulated.pdf", bbox_inches="tight")
    print(f"Saved: {OUTPUT_DIR / 'upset_upregulated.pdf'}")

plt.show()
plt.close(fig)

## 5. Build Consensus Gene Set

A gene enters the **consensus set** if it is significantly upregulated in
at least `MIN_OVERLAP_FRACTION` of the comparisons.

The minimum number of comparisons is computed as:

$$n_{\min} = \lceil \texttt{MIN\_OVERLAP\_FRACTION} \times n_{\text{comparisons}} \rceil$$

- `MIN_OVERLAP_FRACTION = 0.0` → every gene from any comparison (union)
- `MIN_OVERLAP_FRACTION = 1.0` → only genes found in **all** comparisons (strict intersection)
- Intermediate values give a robust consensus tolerant to missing genes in a few comparisons

In [ ]:
# ── Count how many comparisons each gene appears in ──────────────────────────
from collections import Counter

gene_counts = Counter(
    gene for genes in upreg_sets.values() for gene in genes
)

# ── Apply the fractional overlap threshold ────────────────────────────────────
# ceil ensures that fraction=0 maps to min_count=0 (union)
# and fraction=1 maps to min_count=n_comparisons (strict intersection)
min_count = math.ceil(MIN_OVERLAP_FRACTION * n_comparisons)

# Clamp to at least 1 so the union always requires ≥ 1 comparison
min_count = max(min_count, 1)

consensus_genes = sorted(
    gene for gene, cnt in gene_counts.items() if cnt >= min_count
)

print(f"Overlap fraction : {MIN_OVERLAP_FRACTION}")
print(f"Num comparisons  : {n_comparisons}")
print(f"Min count         : {min_count}")
print(f"Consensus genes   : {len(consensus_genes)}")
print()

# Show the full list (or top portion if very large)
if len(consensus_genes) <= 100:
    print("Genes:", ", ".join(consensus_genes))
else:
    print(f"First 100 genes: {', '.join(consensus_genes[:100])}")
    print(f"  … and {len(consensus_genes) - 100} more")

## 6. Over-Representation Analysis (ORA)

For each resource listed in `GENE_SET_RESOURCES` we:
1. Fetch the gene-set network (either via a dedicated `decoupler` getter or from the MSigDB collection hosted on OmniPath)
2. Run Fisher's exact test with `decoupler.mt.query_set` on the consensus gene list
3. Collect the results into one long-form table

**Regulatory / signalling resources** (decoupler built-ins):
- **MSigDB Hallmark** — 50 well-defined biological-state gene sets
- **PROGENy** — pathway-responsive gene signatures
- **CollecTRI** — transcription-factor → target gene regulatory network
- **DoRothEA** — curated TF regulons (confidence levels A–C)

**Pathway / ontology databases** (MSigDB via OmniPath — no extra packages needed):
- **GO Biological Process / Cellular Component / Molecular Function**
- **KEGG Pathways**
- **Reactome**
- **WikiPathways**

> The full MSigDB table (~6 M rows) is downloaded once from OmniPath and
> cached in memory for the duration of the session.

In [ ]:
# ── MSigDB collection helper (cached download) ────────────────────────────────
# dc.op.resource("MSigDB") returns ~6 M rows; we download once and slice by
# collection name to build the net DataFrames that dc.mt.query_set expects.

_msigdb_cache: pd.DataFrame | None = None


def _get_msigdb() -> pd.DataFrame:
    """Download the full MSigDB table from OmniPath (cached after first call)."""
    global _msigdb_cache
    if _msigdb_cache is None:
        print("  Downloading MSigDB from OmniPath (one-time download) …")
        _msigdb_cache = dc.op.resource("MSigDB")
        print(f"  MSigDB table: {len(_msigdb_cache):,} rows")
    return _msigdb_cache


def _msigdb_collection(collection: str, organism: str) -> pd.DataFrame:
    """Extract one MSigDB collection as a source/target net DataFrame."""
    msig = _get_msigdb()
    net = msig.loc[
        msig["collection"] == collection, ["geneset", "genesymbol"]
    ].copy()
    net = net.rename(columns={"geneset": "source", "genesymbol": "target"})
    net = net.drop_duplicates(["source", "target"]).reset_index(drop=True)
    if organism != "human":
        net = dc.op.translate(
            net, columns=["target"], target_organism=organism
        )
    return net


# ── Resource dispatcher ────────────────────────────────────────────────────────
_RESOURCE_GETTERS: dict[str, object] = {
    # Built-in decoupler resources
    "hallmark":  lambda org: dc.op.hallmark(organism=org),
    "progeny":   lambda org: dc.op.progeny(organism=org, top=500),
    "collectri": lambda org: dc.op.collectri(organism=org),
    "dorothea":  lambda org: dc.op.dorothea(organism=org, levels=["A", "B", "C"]),
    # MSigDB collections via OmniPath
    "go_biological_process": lambda org: _msigdb_collection("go_biological_process", org),
    "go_cellular_component": lambda org: _msigdb_collection("go_cellular_component", org),
    "go_molecular_function": lambda org: _msigdb_collection("go_molecular_function", org),
    "kegg_pathways":         lambda org: _msigdb_collection("kegg_pathways", org),
    "reactome_pathways":     lambda org: _msigdb_collection("reactome_pathways", org),
    "wikipathways":          lambda org: _msigdb_collection("wikipathways", org),
}

if not consensus_genes:
    print("⚠  Consensus gene set is empty — skipping enrichment analysis.")
else:
    enrichment_frames: list[pd.DataFrame] = []

    for display_name, resource_key in GENE_SET_RESOURCES.items():
        getter = _RESOURCE_GETTERS.get(resource_key)
        if getter is None:
            print(
                f"⚠  Unknown resource key '{resource_key}' — skipping. "
                f"Available: {list(_RESOURCE_GETTERS.keys())}"
            )
            continue

        print(f"Fetching {display_name} ({resource_key}) …")
        net = getter(ORGANISM)
        print(f"  Network: {len(net):,} rows, {net['source'].nunique():,} sources")

        # Run ORA — returns columns: source, stat, pval, padj
        result = dc.mt.query_set(
            features=consensus_genes,
            net=net,
            verbose=False,
        )

        result["resource"] = display_name
        enrichment_frames.append(result)

        n_sig = int((result["padj"] < ENRICH_PVAL_CUTOFF).sum())
        print(f"  Significant terms (padj < {ENRICH_PVAL_CUTOFF}): {n_sig}")

    if enrichment_frames:
        enrichment_all = pd.concat(enrichment_frames, ignore_index=True)
        print(f"\nTotal enrichment results: {len(enrichment_all):,} rows")
    else:
        enrichment_all = pd.DataFrame()
        print("No enrichment results produced.")

### 6a. Display Top Enriched Terms

Significant enrichment results grouped by resource, sorted by `padj`.
Columns returned by `dc.mt.query_set`: `source` (gene-set name), `stat` (test statistic), `pval`, `padj`.

In [ ]:
if not enrichment_all.empty:
    sig = enrichment_all[enrichment_all["padj"] < ENRICH_PVAL_CUTOFF].copy()

    if sig.empty:
        print("No significantly enriched terms at the chosen FDR threshold.")
    else:
        # Show top 15 terms per resource
        top = sig.sort_values("padj").groupby("resource").head(15)
        display(
            top[["resource", "source", "stat", "pval", "padj"]]
            .style
            .format({"stat": "{:.3f}", "pval": "{:.2e}", "padj": "{:.2e}"})
            .background_gradient(subset="stat", cmap="YlOrRd")
        )
else:
    print("No enrichment results to display.")

## 7. Export Results

Save:
- `consensus_genes.txt` — one gene per line
- `enrichment_all.csv` — full enrichment table across all resources
- `enrichment_significant.csv` — filtered to FDR < `ENRICH_PVAL_CUTOFF`

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Consensus gene list ──────────────────────────────────────────────────────
genes_path = OUTPUT_DIR / "consensus_genes.txt"
genes_path.write_text("\n".join(consensus_genes) + "\n")
print(f"Saved {len(consensus_genes)} consensus genes → {genes_path}")

# ── Full enrichment table ─────────────────────────────────────────────────────
if not enrichment_all.empty:
    all_path = OUTPUT_DIR / "enrichment_all.csv"
    enrichment_all.to_csv(all_path, index=False)
    print(f"Saved full enrichment table → {all_path}")

    sig = enrichment_all[enrichment_all["padj"] < ENRICH_PVAL_CUTOFF]
    if not sig.empty:
        sig_path = OUTPUT_DIR / "enrichment_significant.csv"
        sig.to_csv(sig_path, index=False)
        print(f"Saved significant enrichment table → {sig_path}")
    else:
        print("No significant enrichments to save.")
else:
    print("No enrichment results to save.")

---

## Summary

| Step | Tool / Function | Key parameters |
|------|-----------------|----------------|
| Load DE results | `pd.read_csv()` | `RESULTS_DIR` |
| Filter upregulated | pandas boolean mask | `PVAL_CUTOFF`, `LFC_CUTOFF` |
| UpSet plot | `scutils.pl.upset_plot()` | `sort_by`, `min_subset_size` |
| Consensus set | `collections.Counter` + `math.ceil` | `MIN_OVERLAP_FRACTION` |
| ORA | `dc.mt.query_set()` | `GENE_SET_RESOURCES`, `ORGANISM` |
| Regulatory resources | `dc.op.hallmark / progeny / collectri / dorothea` | `ORGANISM` |
| Pathway resources | `dc.op.resource("MSigDB")` → GO, KEGG, Reactome, WikiPathways | `ORGANISM` |
| Export | `pd.DataFrame.to_csv()`, `Path.write_text()` | `OUTPUT_DIR` |

**To add or remove a gene-set resource:** edit `GENE_SET_RESOURCES` in the
configuration cell.  Supported keys are listed in `_RESOURCE_GETTERS`.

**To change the overlap stringency:** adjust `MIN_OVERLAP_FRACTION`
(`0.0` = union, `1.0` = strict intersection).